# Algo Trading Pipeline -- Colab Runner

Runs `01_Master_Code.py` against your Google Drive folder (`02_Claude_Trading`) instead of the desktop `F:\05_Claude_Automation` path. Code is pulled fresh from GitHub (`harishchinnni-droid/shortsgrapher_tool`) every time you run this notebook, so edits made on the desktop just need a `git push` before your next Colab run.

**Read before running LIVE mode:** Colab notebooks (any tier, including Pro) are not built to be always-on services. Google can disconnect or recycle this session mid-run with no retry, and LIVE mode needs several uninterrupted hours watching open positions. A silent disconnect mid-session means stop-losses stop being checked on real, live positions. This was flagged as a real risk and accepted -- if you ever see a LIVE session drop with a position still open, treat that as the thing to fix next, not a one-off glitch.

**One-time manual step required before your first BACKTEST run:** already done if you've uploaded a real `01_SourceFile.xlsx` into `02_Claude_Trading` (not just the native Google Sheet version).

In [ ]:
# Cell 1 -- Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 2 -- Point the pipeline at your mounted Drive folder.
# Change DRIVE_FOLDER_NAME if your folder inside 'My Drive' has a
# different name than '02_Claude_Trading'.
import os

DRIVE_FOLDER_NAME = "02_Claude_Trading"
os.environ["ALGO_BASE_DIR"] = f"/content/drive/MyDrive/{DRIVE_FOLDER_NAME}"

print("ALGO_BASE_DIR set to:", os.environ["ALGO_BASE_DIR"])
assert os.path.isdir(os.environ["ALGO_BASE_DIR"]), (
    "That folder wasn't found in your mounted Drive -- check the name/path "
    "above matches what you see under 'My Drive' exactly."
)

In [ ]:
# Cell 3 -- Pull the latest code from GitHub. Clones fresh the first time,
# pulls the latest commit on every re-run after that -- so any edit pushed
# from the desktop shows up here automatically without re-uploading files.
REPO_URL = "https://github.com/harishchinnni-droid/shortsgrapher_tool.git"
REPO_DIR = "/content/shortsgrapher_tool"

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

In [ ]:
# Cell 4 -- Install Python dependencies.
!pip install -q -r {REPO_DIR}/requirements.txt

In [ ]:
# Cell 4.5 -- [REVISED] Colab-only browser setup for Zerodha's automated
# Selenium login. First attempt used `chromium-chromedriver` (apt) --
# that combo is broken on Ubuntu 22.04/24.04 (what Colab runs): Ubuntu
# moved chromium-browser to a snap-only package, so the .deb apt installs
# is just a non-functional wrapper script that tries to invoke snap,
# which doesn't exist in Colab's container. chromedriver launches it and
# it immediately exits (status code 1) -- nothing to do with this
# pipeline's own code. Fix: install real Google Chrome from Google's own
# apt repo instead of Chromium -- no snap dependency, this is the
# standard, reliable path for Selenium-in-Colab. CHROMEDRIVER_PATH is
# deliberately NOT set here -- broker_auth.py falls back to
# ChromeDriverManager().install() (already a dependency, used reliably
# elsewhere) which auto-detects the installed Chrome's version and
# downloads a matching driver, avoiding any version-mismatch risk from
# hand-pairing a driver ourselves.
!wget -q -O - https://dl-ssl.google.com/linux/linux_signing_key.pub | apt-key add - > /dev/null 2>&1
!echo "deb http://dl.google.com/linux/chrome/deb/ stable main" > /etc/apt/sources.list.d/google-chrome.list
!apt-get update -qq
!apt-get install -y -qq google-chrome-stable

import shutil

chromium_path = (
    shutil.which("google-chrome-stable")
    or shutil.which("google-chrome")
    or "/usr/bin/google-chrome-stable"
)
os.environ["CHROME_BINARY_LOCATION"] = chromium_path
os.environ.pop("CHROMEDRIVER_PATH", None)  # let ChromeDriverManager pick a matching driver

print("chrome binary:", chromium_path, "-- exists:", os.path.exists(chromium_path))
if os.path.exists(chromium_path):
    !{chromium_path} --version

In [ ]:
# Cell 5 -- Run the pipeline. Same interactive prompts as the desktop
# version (LIVE vs BACKTEST, then a date or date range) -- Colab's output
# cell accepts typed input the same way a terminal does.
#
# Uses runpy rather than a plain import: Python module names can't start
# with a digit, so `import 01_Master_Code` is invalid syntax -- this only
# ever worked on desktop because the file is run directly
# (`python 01_Master_Code.py`), never imported. runpy.run_path executes
# the file the same way that command line invocation does.
import sys
if REPO_DIR not in sys.path:
    sys.path.append(REPO_DIR)

import runpy
runpy.run_path(f"{REPO_DIR}/01_Master_Code.py", run_name="__main__")